In [ ]:
import sys
import pandas as pd
import re
from sklearn.model_selection import train_test_split

# Import kamus slang dari src/dictionary.py
sys.path.append('../src')
from dictionary import kamus_slang

In [ ]:
# Load dataset
df = pd.read_csv('../data/dataset_ulasan.csv')

# Drop missing value & duplikasi
df = df.dropna(subset=['teks'])
df = df.drop_duplicates(subset=['teks'])

In [ ]:
def clean_and_normalize(text):
    # 1. Cleaning: Hapus URL, mention, hashtag, angka, dan karakter khusus (pertahankan spasi)
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    text = re.sub(r'[@#]\w+', '', text)
    text = re.sub(r'[^a-zA-Z\s]', ' ', text)
    
    # 2. Lowercase
    text = text.lower()
    
    # 3. Normalisasi Slang
    words = text.split()
    words = [kamus_slang.get(word, word) for word in words]
    
    return " ".join(words)

df['clean_text'] = df['teks'].apply(clean_and_normalize)

In [ ]:
# Format untuk masukan IndoBERT ABSA
df['absa_input'] = df['aspect'] + " [SEP] " + df['clean_text']

# Split data Train & Test terlebih dahulu sebelum Encoding
X_train, X_test, y_train, y_test = train_test_split(
    df['absa_input'], df['label'], test_size=0.2, random_state=42, stratify=df['label']
)

# Simpan hasil preprocessing ke folder data
df.to_csv('../data/clean_data.csv', index=False)
print("Preprocessing Selesai dan Data Berhasil Disimpan!")